# Lab 2 — Depth, Degradation, and Residual Connections

**Unit.** `convnets` · **Duration.** 1 hour

Three networks, built from the same convolutional block and trained with identical
settings. The only thing that changes is what happens inside a stage.

1. A **shallow** convnet — three convolutions. It trains easily.
2. A **deep plain** convnet — nine convolutions, same widths, same pooling. It does not.
3. The **same nine convolutions with skip connections**, then with batch normalization.

The point of the lab is the second one. A deeper network that performs worse **on the
data it was fitted to** is not overfitting; it is a failure to optimise, and it is what
residual connections exist to remove.

Work through the cells in order. Each cell marked **Your turn** is followed by a CHECK
cell that prints `[ok]` when you have it right.

In [ ]:
# COLAB SETUP -- run this cell first.  Outside Colab it does nothing.
import importlib.util
import subprocess
import sys
import urllib.request

try:  # find_spec raises when the parent package is absent, so import instead
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Colab already ships torch and torchvision; install only what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "lightning", "torchmetrics"], check=True)

# Fetch the lab's support module when it is not already beside the notebook.
if importlib.util.find_spec("lab2_utils") is None:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/kenpu-uoit/csci4052u-machine-learning-2/main/labs/lab2/lab2_utils.py", "lab2_utils.py")
    print("downloaded lab2_utils.py")


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
import lightning as L

import lab2_utils
from lab2_utils import (
    MNISTDataModule, LitClassifier, fit, make_trainer,
    show_digits, plot_curves, summary_table, first_conv_grad_norm,
    n_conv, CHANCE_LOSS, EPOCHS,
)

lab2_utils.quiet_lightning()
L.seed_everything(0, workers=True)

dm = MNISTDataModule()
dm.prepare_data()
dm.setup()
show_digits(dm)
print(f"chance-level loss for 10 classes: ln 10 = {CHANCE_LOSS:.4f}")

---
## Exercise 1 · A convolutional block, and a shallow convnet
*15 minutes · outcome `extend-to-2d`*

Every network in this lab is built from one block: a 3x3 convolution that preserves the
spatial size, a ReLU, and a max-pool that halves it. Stacking three of them takes a
28x28 image to 3x3, through 14 and 7.

### 1a · The block — **Your turn**

Write `conv_block(c_in, c_out)` returning an `nn.Sequential` of

* `nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)` — `padding=1` is what keeps the
  spatial size unchanged for a 3x3 kernel,
* `nn.ReLU()`,
* `nn.MaxPool2d(2)`.

In [ ]:
def conv_block(c_in, c_out):
    """3x3 convolution (padding 1) -> ReLU -> 2x2 max pool."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
lab2_utils.check_conv_block(conv_block)

### 1b · The shallow network — **Your turn**

Assemble `ShallowCNN` as an `nn.Sequential`: three blocks widening the channels
`1 -> 16 -> 32 -> 64`, then `nn.Flatten()`, then `nn.Linear(64 * 3 * 3, 10)`.

The head takes `64 * 3 * 3 = 576` features because three poolings take 28 to 3.

In [ ]:
def ShallowCNN():
    """Three convolutions, then a linear head over the 64 x 3 x 3 feature map."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
lab2_utils.check_shallow(ShallowCNN())

### 1c · Training it — **Your turn**

`LitClassifier` provides everything except `training_step`. Implement it: unpack the
batch, run the model, return the cross-entropy loss. Log it as `train_loss` so the
history is recorded.

```python
x, y = batch
logits = self(x)
loss = F.cross_entropy(logits, y)
self.log("train_loss", loss)
return loss
```

Then the cell fits for 5 epochs and plots the loss.

In [ ]:
class MyClassifier(LitClassifier):
    def training_step(self, batch, batch_idx):
        # YOUR CODE HERE
        raise NotImplementedError()


shallow = MyClassifier(ShallowCNN())
shallow_hist, shallow_test_acc = fit(shallow, dm)
plot_curves({"shallow (3 conv)": shallow_hist}, title="shallow network")

In [ ]:
lab2_utils.check_shallow_training(shallow_hist, shallow_test_acc)

---
## Exercise 2 · Depth without help
*20 minutes · outcome `diagnose-deep-network-training`*

Now make the network deeper and change nothing else. Same widths, same pooling, same
optimiser, same number of epochs, same seed. Only the number of convolutions per stage
goes from one to three.

### 2a · The deep plain network — **Your turn**

Build `PlainDeepCNN`: for each of the three stages `(1, 16)`, `(16, 32)`, `(32, 64)`,
emit **three** `Conv2d(..., kernel_size=3, padding=1)` + `ReLU` pairs — the first changes
the channel count, the next two keep it — then a single `MaxPool2d(2)` at the end of the
stage. Finish with `nn.Flatten()` and `nn.Linear(64 * 3 * 3, 10)`.

Nine convolutions, three max-pools.

In [ ]:
def PlainDeepCNN():
    """Nine convolutions: three per stage, one pool per stage."""
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
lab2_utils.check_plain_deep(PlainDeepCNN())

### 2b · Train it the same way — **Your turn**

Wrap it in `MyClassifier`, fit it with `fit(...)` exactly as before, and plot both
curves together. `plot_curves` draws the chance line at `ln 10` for you.

In [ ]:
plain = MyClassifier(PlainDeepCNN())
plain_hist, plain_test_acc = fit(plain, dm)

# YOUR CODE HERE -- call plot_curves with BOTH histories, keyed by name
raise NotImplementedError()

In [ ]:
lab2_utils.check_depth_comparison(shallow_hist, plain_hist)

### 2c · Read the result — **Your turn**

Pull the final **training** loss and **training** accuracy out of each history
(`hist.losses[-1]`, `hist.accs[-1]`) and pass them to the check.

Training, not test. That distinction is the whole exercise: a model that overfits fits
its training set *better* than a smaller one. This one fits it worse.

In [ ]:
shallow_train_loss = shallow_hist.losses[-1]
# YOUR CODE HERE -- set plain_train_loss and plain_train_acc from plain_hist,
# then print both networks' training loss and accuracy
raise NotImplementedError()

In [ ]:
lab2_utils.check_degradation(shallow_train_loss, plain_train_loss, plain_train_acc)

---
## Exercise 3 · Residual connections, then batch normalization
*20 minutes · outcome `build-residual-blocks`*

The nine convolutions are not the problem. How they are wired is. A residual block
computes `z + f(z)` instead of `f(z)`, so a block that has learned nothing passes its
input through untouched rather than destroying it.

### 3a · The block — **Your turn**

Implement `ResidualBlock(c)`, preserving the channel count:

* `conv1`, `conv2`: `nn.Conv2d(c, c, kernel_size=3, padding=1)`,
* forward: `relu(z + conv2(relu(conv1(z))))`.

The check zeroes `conv2` and confirms the block then returns its input unchanged. A
block missing the `z +` cannot pass that test.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, z):
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
lab2_utils.check_residual_block(ResidualBlock)

### 3b · The residual network — **Your turn**

Write `make_residual_net(block_cls)` so the same function serves this exercise and the
next. Each stage is: one `Conv2d` + `ReLU` that changes the channel count, then one
`block_cls(c_out)`, then `MaxPool2d(2)`.

That is 1 + 2 = **three convolutions per stage, nine in total** — exactly the plain
network's count. The only difference is the skip.

In [ ]:
def make_residual_net(block_cls):
    """Nine convolutions, with the middle two of each stage wrapped in a block."""
    # YOUR CODE HERE
    raise NotImplementedError()


residual = MyClassifier(make_residual_net(ResidualBlock))
residual_hist, residual_test_acc = fit(residual, dm)
plot_curves(
    {"shallow (3 conv)": shallow_hist, "plain deep (9 conv)": plain_hist,
     "residual (9 conv)": residual_hist},
    title="the same nine convolutions, wired two ways",
)

In [ ]:
lab2_utils.check_residual_training(
    make_residual_net(ResidualBlock), residual_hist, residual_test_acc, plain_train_loss
)

### 3c · Batch normalization, and the mechanism — **Your turn**

Two things.

**First**, write `ResidualBlockBN(c)` in the pre-activation order the notes give:
`BN -> ReLU -> Conv`, twice, then add the input. Note there is no ReLU on the output, so
the block can subtract as well as add.

**Second**, measure *why* it works. `first_conv_grad_norm(net, batch)` runs one backward
pass and returns the gradient norm at the first convolution. Measure it for a fresh
plain network and a fresh residual network, both at initialisation, on the same batch.

In [ ]:
class ResidualBlockBN(nn.Module):
    def __init__(self, c):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, z):
        # YOUR CODE HERE
        raise NotImplementedError()


residual_bn = MyClassifier(make_residual_net(ResidualBlockBN))
bn_hist, bn_test_acc = fit(residual_bn, dm)

batch = next(iter(dm.train_dataloader()))
L.seed_everything(0, workers=True)
plain_norm = first_conv_grad_norm(PlainDeepCNN(), batch)
L.seed_everything(0, workers=True)
res_norm = first_conv_grad_norm(make_residual_net(ResidualBlock), batch)

rows = [
    ("shallow (3 conv)", shallow_hist.losses[-1], shallow_hist.accs[-1], shallow_test_acc),
    ("plain deep (9 conv)", plain_hist.losses[-1], plain_hist.accs[-1], plain_test_acc),
    ("residual (9 conv)", residual_hist.losses[-1], residual_hist.accs[-1], residual_test_acc),
    ("residual + BN (9 conv)", bn_hist.losses[-1], bn_hist.accs[-1], bn_test_acc),
]
summary_table(rows)
print(f"\nfirst-conv gradient norm at init: plain {plain_norm:.3e}, residual {res_norm:.3e}")

In [ ]:
lab2_utils.check_batchnorm_and_flow(
    rows, plain_norm, res_norm, bn_test_acc, residual_test_acc
)

---
## What this lab did not cover

The `convnets` unit is wider than one hour. Out of scope here, and worth reading in the
notes:

* **1D convolution**, and convolution as a banded, weight-tied fully connected layer.
* **Stride, dilation and padding arithmetic** — the output-size formula.
* **Receptive fields** through a stack, and how depth buys context.
* **Upsampling**: duplication, max unpooling, bilinear interpolation, transposed convolution.
* **1x1 convolutions** for changing channel depth without touching resolution.
* **DenseNet and U-Net** — concatenation instead of addition, and long skips.
* **Vision tasks** — detection and segmentation, and the architectures each needs.

What this lab did establish is the one result those all build on: past a certain depth, a
plain stack stops training, and an additive skip connection restores it.